# 00 · Data Loading & Standardization (5-Model Pipeline)
Loads the four delivered datasets, canonicalizes county names, rejects bad
sources (e.g. the old hourly MET file), and writes clean `_FINAL.csv` artifacts.

These standardized datasets feed Notebook 01 (Master Merge & Target Synthesis),
which in turn supports the 5-model pipeline: Logistic Regression, Random Forest,
XGBoost, LightGBM, and Linear Regression.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../../data_pipeline/data/processed")

CANONICAL_COUNTIES = [
    "Baringo","Bomet","Bungoma","Busia","Elgeyo Marakwet","Embu","Garissa",
    "Homa Bay","Isiolo","Kajiado","Kakamega","Kericho","Kiambu","Kilifi",
    "Kirinyaga","Kisii","Kisumu","Kitui","Kwale","Laikipia","Lamu","Machakos",
    "Makueni","Mandera","Marsabit","Meru","Migori","Mombasa","Murang'a",
    "Nairobi","Nakuru","Nandi","Narok","Nyamira","Nyandarua","Nyeri",
    "Samburu","Siaya","Taita Taveta","Tana River","Tharaka Nithi","Trans Nzoia",
    "Turkana","Uasin Gishu","Vihiga","Wajir","West Pokot",
]

def norm(name: str) -> str:
    return (str(name).lower().replace(" ", "").replace("-", "")
            .replace("'", "").replace("'", ""))

LOOKUP = {norm(c): c for c in CANONICAL_COUNTIES}

NDVI_COLS = ["county","month","ndvi_mean","month_num","month_sin","month_cos",
             "ndvi_lag1","ndvi_lag3","ndvi_roll3_mean","ndvi_roll3_std",
             "ndvi_anomaly","ndvi_mom_change"]

MET_COLS = ["county","year","month","temp_mean_c","rainfall_mm","soil_moisture"]
MET_RENAME = {"temperature_mean":"temp_mean_c","rainfall_sum":"rainfall_mm",
              "soil_moisture_mean":"soil_moisture"}

In [ ]:
def first_nonempty(candidates):
    for p in candidates:
        if p.exists() and p.stat().st_size > 0:
            return p
    raise FileNotFoundError(f"No non-empty file among: {[str(p) for p in candidates]}")

def read_csv_auto(path, cols, rename=None):
    """Read a CSV whether or not it has a header row."""
    with open(path, "r") as f:
        first_cell = f.readline().split(",")[0].strip().lower()
    if first_cell == "county":                      # real header present
        df = pd.read_csv(path, skip_blank_lines=True)
        if rename: df = df.rename(columns=rename)
        df = df.reindex(columns=cols)
    else:                                           # headerless fallback
        df = pd.read_csv(path, header=None, names=cols, skip_blank_lines=True)
    return df.dropna(how="all").reset_index(drop=True)

def map_county(s): return s.map(lambda c: LOOKUP.get(norm(c), c))
def unmapped(df): return df.loc[~df["county"].isin(CANONICAL_COUNTIES), "county"].unique()

In [ ]:
ndvi = read_csv_auto(first_nonempty([DATA_DIR / "NDVI_cleaned.csv"]), NDVI_COLS)
ndvi["county"] = map_county(ndvi["county"])
ndvi[NDVI_COLS[2:]] = ndvi[NDVI_COLS[2:]].apply(pd.to_numeric, errors="coerce")
ndvi = (ndvi.dropna(subset=["county","month"])
            .drop_duplicates(["county","month"])
            .sort_values(["county","month"]))
ndvi.to_csv(DATA_DIR / "NDVI_cleaned_FINAL.csv", index=False)
ndvi.head()

In [ ]:
import io, re

def load_met_robust(path):
    """Parse the monthly MET file even if rows were glued together."""
    raw = Path(path).read_text()
    # 1) Header glued to first data row: "...soil_moisture_meanBaringo,2020,1"
    raw = raw.replace("soil_moisture_mean", "soil_moisture_mean\n")
    # 2) Data rows glued: "...0.11944Baringo,2020,1" -> newline before county capital.
    #    NOTE: do NOT use a lowercase->uppercase regex here; it splits
    #    camelCase names like TransNzoia / UasinGishu mid-name.
    raw = re.sub(r"([0-9])([A-Z][a-z])", r"\1\n\2", raw)
    df = pd.read_csv(io.StringIO(raw))
    df = df.rename(columns=MET_RENAME)
    return df.reindex(columns=MET_COLS)

search_dirs = [DATA_DIR, DATA_DIR.parent, Path("../.."), Path.home() / "Downloads"]
names = ["Metrological Cleaned Data 2.csv",
         "Metrological_Cleaned_Data_2.csv",
         "Metrological_Cleaned_Data.csv",
         "Metrological_Data_cleaned_2.csv",
         "Metrological_Data_cleaned.csv",
         "Pasted_Text_1788358853270.txt"]

met, used = None, None
for d in search_dirs:
    for n in names:
        p = d / n
        if not (p.exists() and p.stat().st_size > 0):
            continue
        try:
            df = load_met_robust(p)
        except Exception as e:
            print(f"⚠ Parse failed for {p}: {e}")
            continue
        yr = pd.to_numeric(df["year"],  errors="coerce")
        mo = pd.to_numeric(df["month"], errors="coerce")
        valid = yr.between(2019, 2026) & mo.between(1, 12)
        good = valid.mean()
        if good >= 0.9:
            met = df[valid].copy()          # keep only sane rows
            met["year"]  = met["year"].astype(int)
            met["month"] = met["month"].astype(int)
            used = p
            break
        print(f"⚠ Rejected {p.name}: only {good:.0%} valid rows (hourly file?)")
    if met is not None:
        break

if met is None:
    raise FileNotFoundError("No valid monthly MET file found among candidates.")

print(f"✅ Using MET source: {used}  ({len(met)} rows)")

In [ ]:
demo = pd.read_csv(first_nonempty([DATA_DIR / "Demographic_Crop Data_cleaned.csv",
                                   DATA_DIR / "Demographic_Crop_Data_cleaned.csv"]))
demo.columns = [c.strip().lower().replace(" ", "_") for c in demo.columns]
demo["county"] = map_county(demo["county"])
num_cols = [c for c in demo.columns if c != "county"]

# Clean column-by-column (DataFrame has no .str accessor, Series does)
for c in num_cols:
    demo[c] = demo[c].astype(str).str.replace(",", "", regex=False).str.strip()

# Now convert to numbers ("378422" -> 378422; anything unparseable -> NaN)
demo[num_cols] = demo[num_cols].apply(pd.to_numeric, errors="coerce")
demo = demo.dropna(subset=["county"])
demo.to_csv(DATA_DIR / "Demographic_FINAL.csv", index=False)
demo.head()

In [ ]:
hist = pd.read_csv(first_nonempty([DATA_DIR / "Historic Crop Production_cleaned.csv",
                                   DATA_DIR / "Historic_Crop_Production_cleaned.csv"]))
hist["Commodity"] = hist["Commodity"].str.strip()

seen = {}
def fix_total(c):
    if c.lower() == "total":
        seen[c] = seen.get(c, 0) + 1
        return f"Total_{seen[c]}"
    return c
hist["Commodity"] = hist["Commodity"].map(fix_total)

year_cols = [c for c in hist.columns if str(c).startswith("20")]
hist_long = hist.melt(id_vars="Commodity", value_vars=year_cols,
                      var_name="year", value_name="value")
hist_long["year"] = hist_long["year"].astype(str).str.rstrip("*").astype(int)
hist_long.to_csv(DATA_DIR / "Crop_Production_LONG_FINAL.csv", index=False)
hist_long.head()

In [ ]:
# ---- Complete MET processing ----
met["county"] = map_county(met["county"])                    # TransNzoia -> Trans Nzoia, etc.
met = met.dropna(subset=["county", "year", "month"])
met = met.drop_duplicates(subset=["county", "year", "month"])
met["month_key"] = met["year"].astype(str) + "-" + met["month"].astype(str).str.zfill(2)
met = met.sort_values(["county", "month_key"])
met.to_csv(DATA_DIR / "Metrological_Cleaned_Data_2_FINAL.csv", index=False)
print(f"MET final: {len(met)} rows, {met['county'].nunique()} counties")
met.head()

In [ ]:
print("=== NDVI ===")
print(f"rows={len(ndvi)}  counties={ndvi['county'].nunique()}  "
      f"range {ndvi['month'].min()} -> {ndvi['month'].max()}  nulls={int(ndvi.isna().sum().sum())}")
print("unmapped:", unmapped(ndvi) if len(unmapped(ndvi)) else "none")

print("\n=== MET ===")
print(f"rows={len(met)}  counties={met['county'].nunique()}  "
      f"range {met['month_key'].min()} -> {met['month_key'].max()}  nulls={int(met.isna().sum().sum())}")
print("unmapped:", unmapped(met) if len(unmapped(met)) else "none")

print("\n=== DEMOGRAPHIC ===")
missing = sorted(set(CANONICAL_COUNTIES) - set(demo["county"]))
print(f"rows={len(demo)}  missing counties: {missing if missing else 'none'}")

print("\n=== NDVI ∩ MET overlap ===")
ndvi_keys, met_keys = set(zip(ndvi["county"], ndvi["month"])), set(zip(met["county"], met["month_key"]))
print(f"{len(ndvi_keys & met_keys)}/{len(ndvi_keys)} NDVI rows will get weather features "
      f"({100*len(ndvi_keys & met_keys)/len(ndvi_keys):.1f}%)")